# 19.12 — Membership Inference

Membership inference asks a simple privacy question: given a trained model, an example `(x, y)`, and the model's score on that example, can an attacker tell whether that example was in the training set? In this lesson we build the whole audit from scratch in NumPy: a toy classifier, the train/test loss gap that creates leakage, a loss-threshold attack, shadow models that choose the threshold without touching the victim's data, and a likelihood-ratio attack (LiRA) that compares how surprising a score is under member versus non-member behavior.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build membership inference one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is made visible so the attack is an auditable calculation rather than a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random simulation, and linear algebra for the toy privacy audit.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for data, model fitting, and attack splits.

### 1. A toy classifier whose training examples get lower loss

Membership inference starts with a model quantity that behaves differently on training points than on fresh points. We will use per-example logistic loss, because it is small when the model assigns high probability to the true label and large when the model is uncertain or wrong. A model that fits its training set too tightly often gives members unusually low losses — an overfitting fingerprint.

In [ ]:
def sigmoid_w(z_w):
    return 1 / (1 + np.exp(-np.clip(z_w, -30, 30)))

def make_blobs_w(n_w, seed_w, noise_w=0.9):
    rng_w = np.random.default_rng(seed_w)
    y_w = rng_w.integers(0, 2, size=n_w)
    centers_w = np.where(y_w[:, None] == 1, np.array([1.0, 1.0]), np.array([-1.0, -1.0]))
    X_w = centers_w + noise_w * rng_w.normal(size=(n_w, 2))
    return X_w, y_w.astype(float)

X_train_w, y_train_w = make_blobs_w(80, 1)
X_test_w, y_test_w = make_blobs_w(400, 2)

print("train shape:", X_train_w.shape, "test shape:", X_test_w.shape)
print("train class rate:", round(float(y_train_w.mean()), 3))

▶ What you'll see: a small training set and a larger fresh test set drawn from the same two-class toy distribution.

In [ ]:
def fit_logreg_w(X_w, y_w, steps_w=2500, lr_w=0.25, lam_w=0.001):
    X1_w = np.c_[np.ones(len(X_w)), X_w]
    theta_w = np.zeros(X1_w.shape[1])
    for _ in range(steps_w):
        p_w = sigmoid_w(X1_w @ theta_w)
        grad_w = X1_w.T @ (p_w - y_w) / len(y_w) + lam_w * np.r_[0, theta_w[1:]]
        theta_w -= lr_w * grad_w
    return theta_w

def predict_proba_w(X_w, theta_w):
    return sigmoid_w(np.c_[np.ones(len(X_w)), X_w] @ theta_w)

theta_w = fit_logreg_w(X_train_w, y_train_w)
train_p_w = predict_proba_w(X_train_w, theta_w)
test_p_w = predict_proba_w(X_test_w, theta_w)

print("theta:", np.round(theta_w, 3))
print("train accuracy:", round(float(np.mean((train_p_w >= 0.5) == y_train_w)), 3))
print("test accuracy:", round(float(np.mean((test_p_w >= 0.5) == y_test_w)), 3))

▶ What you'll see: a useful classifier whose train and test accuracies are similar but not identical.

In [ ]:
def log_loss_w(p_w, y_w):
    eps_w = 1e-12
    p_w = np.clip(p_w, eps_w, 1 - eps_w)
    return -(y_w * np.log(p_w) + (1 - y_w) * np.log(1 - p_w))

train_loss_w = log_loss_w(train_p_w, y_train_w)
test_loss_w = log_loss_w(test_p_w, y_test_w)

print("mean train loss:", round(float(train_loss_w.mean()), 3))
print("mean test loss:", round(float(test_loss_w.mean()), 3))

assert train_loss_w.mean() < test_loss_w.mean()

▶ What you'll see: the training loss is lower than the test loss, which is exactly the separation a membership attacker tries to exploit.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(train_loss_w, bins=22, alpha=0.65, label="members: train loss", color="seagreen")
plt.hist(test_loss_w, bins=22, alpha=0.55, label="non-members: test loss", color="crimson")
plt.xlabel("per-example logistic loss")
plt.ylabel("count")
plt.title("1: member vs non-member loss histograms")
plt.legend()
plt.show()

▶ What you'll see: the member histogram leans left. The overlap is why the attack is imperfect; the left shift is why the attack is possible.

*Why it's done this way:* Logistic loss is a calibrated way to inspect confidence on the true label: for a correct label with probability `p`, the loss is `-log(p)`, so moving from `p=0.9` to `p=0.99` visibly lowers loss. Training minimizes average loss on members, so overfitting naturally creates a loss gap even when raw accuracy looks fine. The attacker does not need to know the true training algorithm's internals; it only needs a score whose distribution differs for members and non-members.

### 2. The loss-threshold attack and attack advantage

The simplest attack is the rule from the source lesson: $$\hat m=\mathbf{1}[s(x,y)\ge \tau]$$. If we let the score be negative loss, `s = -loss`, then high score means "the model is unusually confident on this labeled example." Equivalently, we can threshold loss directly and call low-loss points members.

In [ ]:
member_scores_w = -train_loss_w
nonmember_scores_w = -test_loss_w
all_scores_w = np.r_[member_scores_w, nonmember_scores_w]
all_m_w = np.r_[np.ones_like(member_scores_w), np.zeros_like(nonmember_scores_w)]

print("member score mean:", round(float(member_scores_w.mean()), 3))
print("non-member score mean:", round(float(nonmember_scores_w.mean()), 3))

▶ What you'll see: members have larger scores because their losses are smaller.

In [ ]:
thresholds_w = np.linspace(all_scores_w.min(), all_scores_w.max(), 200)
best_adv_w, best_tau_w, best_acc_w = -1, None, None
for tau_w in thresholds_w:
    pred_m_w = (all_scores_w >= tau_w).astype(float)
    tpr_w = np.mean(pred_m_w[all_m_w == 1] == 1)
    fpr_w = np.mean(pred_m_w[all_m_w == 0] == 1)
    adv_w = tpr_w - fpr_w
    acc_w = np.mean(pred_m_w == all_m_w)
    if adv_w > best_adv_w:
        best_adv_w, best_tau_w, best_acc_w = adv_w, tau_w, acc_w

print("best score threshold tau:", round(float(best_tau_w), 3))
print("attack accuracy:", round(float(best_acc_w), 3))
print("attack advantage TPR-FPR:", round(float(best_adv_w), 3))

assert best_adv_w > 0.10

▶ What you'll see: the threshold attack beats random guessing by a measurable advantage.

In [ ]:
tpr_curve_w, fpr_curve_w = [], []
for tau_w in thresholds_w:
    pred_m_w = (all_scores_w >= tau_w).astype(float)
    tpr_curve_w.append(np.mean(pred_m_w[all_m_w == 1] == 1))
    fpr_curve_w.append(np.mean(pred_m_w[all_m_w == 0] == 1))
order_w = np.argsort(fpr_curve_w)
auc_w = float(np.trapz(np.array(tpr_curve_w)[order_w], np.array(fpr_curve_w)[order_w]))

print("ROC AUC:", round(auc_w, 3))

assert auc_w > 0.53
plt.figure(figsize=(4.5, 3.5))
plt.plot(np.array(fpr_curve_w)[order_w], np.array(tpr_curve_w)[order_w], color="navy", label=f"AUC={auc_w:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="random")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("2: ROC for the loss-threshold attack")
plt.legend()
plt.show()

▶ What you'll see: the ROC curve sits above the diagonal. Look for how far it bows upward; that distance is privacy leakage.

*Why it's done this way:* Raw attack accuracy can be misleading when membership labels are imbalanced, so `TPR - FPR` measures the attack's advantage over a false-alarm baseline. The threshold is chosen on attack calibration data, not on the victim point being judged, because otherwise the attacker would overfit the audit itself. ROC repeats the same threshold rule over all possible `τ` values, showing the full privacy tradeoff instead of one cherry-picked operating point.

### 3. Shadow models estimate the threshold without seeing victim membership

A realistic attacker may not know which examples were in the victim model's training set. Shadow models solve that calibration problem: train many lookalike models on data from the same population, record each shadow model's losses on its own training data (known members) and held-out data (known non-members), and choose an attack threshold from those labeled shadow losses.

In [ ]:
def shadow_losses_w(seed_w):
    Xs_w, ys_w = make_blobs_w(80, seed_w)
    Xh_w, yh_w = make_blobs_w(200, seed_w + 1000)
    th_w = fit_logreg_w(Xs_w, ys_w, steps_w=1400, lr_w=0.25, lam_w=0.001)
    in_loss_w = log_loss_w(predict_proba_w(Xs_w, th_w), ys_w)
    out_loss_w = log_loss_w(predict_proba_w(Xh_w, th_w), yh_w)
    return in_loss_w, out_loss_w

shadow_in_w, shadow_out_w = [], []
for seed_w in range(10, 18):
    a_w, b_w = shadow_losses_w(seed_w)
    shadow_in_w.append(a_w)
    shadow_out_w.append(b_w)
shadow_in_w = np.concatenate(shadow_in_w)
shadow_out_w = np.concatenate(shadow_out_w)

print("shadow member losses:", shadow_in_w.shape[0])
print("shadow non-member losses:", shadow_out_w.shape[0])

▶ What you'll see: the attacker has many labeled member/non-member loss samples from its own shadow experiments.

In [ ]:
shadow_scores_w = np.r_[-shadow_in_w, -shadow_out_w]
shadow_m_w = np.r_[np.ones_like(shadow_in_w), np.zeros_like(shadow_out_w)]
shadow_taus_w = np.linspace(shadow_scores_w.min(), shadow_scores_w.max(), 200)
shadow_adv_w = []
for tau_w in shadow_taus_w:
    pred_w = (shadow_scores_w >= tau_w).astype(float)
    shadow_adv_w.append(np.mean(pred_w[shadow_m_w == 1] == 1) - np.mean(pred_w[shadow_m_w == 0] == 1))
shadow_tau_w = float(shadow_taus_w[int(np.argmax(shadow_adv_w))])

print("shadow-chosen tau:", round(shadow_tau_w, 3))

▶ What you'll see: a threshold chosen without using the victim model's true membership labels.

In [ ]:
victim_pred_w = (all_scores_w >= shadow_tau_w).astype(float)
victim_tpr_w = np.mean(victim_pred_w[all_m_w == 1] == 1)
victim_fpr_w = np.mean(victim_pred_w[all_m_w == 0] == 1)

print("victim TPR:", round(float(victim_tpr_w), 3))
print("victim FPR:", round(float(victim_fpr_w), 3))
print("victim advantage:", round(float(victim_tpr_w - victim_fpr_w), 3))

assert victim_tpr_w - victim_fpr_w > 0.05
plt.figure(figsize=(5, 3))
plt.hist(shadow_in_w, bins=25, alpha=0.65, label="shadow members", color="seagreen")
plt.hist(shadow_out_w, bins=25, alpha=0.5, label="shadow non-members", color="crimson")
plt.axvline(-shadow_tau_w, color="black", linestyle="--", label="chosen loss cutoff")
plt.xlabel("loss")
plt.title("3: shadow losses calibrate the attack")
plt.legend()
plt.show()

▶ What you'll see: the vertical cutoff is learned from shadows, then transferred to the victim. It should land where member losses are relatively dense and non-member losses thin out.

*Why it's done this way:* Shadow models convert a privacy question with unknown labels into a supervised calibration problem. The attacker cannot train on the victim's membership labels, but it can imitate the data distribution and training recipe to estimate what member and non-member scores tend to look like. This matters because choosing `τ` on the victim audit set would inflate attack performance, exactly like tuning a model on its test set.

### 4. LiRA: a likelihood-ratio view of membership evidence

LiRA (Likelihood Ratio Attack) goes beyond one global threshold. Instead of asking only whether loss is below a cutoff, it asks: "is this score more likely under the member-score distribution or under the non-member-score distribution?" In the simplest Gaussian version, we estimate two normal distributions for the score `s=-loss` and compute a log likelihood ratio.

In [ ]:
def normal_logpdf_w(x_w, mu_w, sd_w):
    sd_w = max(float(sd_w), 1e-6)
    return -0.5 * np.log(2 * np.pi * sd_w ** 2) - 0.5 * ((x_w - mu_w) / sd_w) ** 2

mu_in_w, raw_sd_in_w = float(shadow_scores_w[shadow_m_w == 1].mean()), float(shadow_scores_w[shadow_m_w == 1].std())
mu_out_w, raw_sd_out_w = float(shadow_scores_w[shadow_m_w == 0].mean()), float(shadow_scores_w[shadow_m_w == 0].std())
sd_pool_w = float(np.sqrt((raw_sd_in_w ** 2 + raw_sd_out_w ** 2) / 2))
sd_in_w, sd_out_w = sd_pool_w, sd_pool_w

print("member score Gaussian: mean", round(mu_in_w, 3), "sd", round(sd_in_w, 3))
print("non-member score Gaussian: mean", round(mu_out_w, 3), "sd", round(sd_out_w, 3))

assert mu_in_w > mu_out_w

▶ What you'll see: the member-score distribution is shifted upward because members tend to have lower loss.

In [ ]:
llr_w = normal_logpdf_w(all_scores_w, mu_in_w, sd_in_w) - normal_logpdf_w(all_scores_w, mu_out_w, sd_out_w)

print("mean member LLR:", round(float(llr_w[all_m_w == 1].mean()), 3))
print("mean non-member LLR:", round(float(llr_w[all_m_w == 0].mean()), 3))

assert llr_w[all_m_w == 1].mean() > llr_w[all_m_w == 0].mean()

▶ What you'll see: members have larger log likelihood ratios, meaning their scores look more member-like under the shadow distributions.

In [ ]:
llr_taus_w = np.linspace(llr_w.min(), llr_w.max(), 200)
llr_tpr_w, llr_fpr_w = [], []
for tau_w in llr_taus_w:
    pred_w = (llr_w >= tau_w).astype(float)
    llr_tpr_w.append(np.mean(pred_w[all_m_w == 1] == 1))
    llr_fpr_w.append(np.mean(pred_w[all_m_w == 0] == 1))
llr_order_w = np.argsort(llr_fpr_w)
llr_auc_w = float(np.trapz(np.array(llr_tpr_w)[llr_order_w], np.array(llr_fpr_w)[llr_order_w]))

print("LiRA-style AUC:", round(llr_auc_w, 3))

assert llr_auc_w > 0.53
plt.figure(figsize=(4.5, 3.5))
plt.plot(np.array(llr_fpr_w)[llr_order_w], np.array(llr_tpr_w)[llr_order_w], color="purple", label=f"LiRA AUC={llr_auc_w:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("4: ROC using likelihood-ratio scores")
plt.legend()
plt.show()

▶ What you'll see: a ROC curve from likelihood ratios. Look for whether it improves or matches the simple loss-threshold curve.

*Why it's done this way:* A likelihood ratio is the Bayesian evidence update in log form: positive values mean the observed score is more probable if the point was a member than if it was not. LiRA is powerful because it can use distribution shape and, in stronger versions, per-example or per-class calibration; a single threshold assumes all examples share the same score distribution. Here we keep the Gaussian version small so the math is visible: log density under "in" minus log density under "out."

### 5. Overfitting is the leak amplifier, and regularization is a privacy knob

The same attack can be rerun after changing the model's capacity or regularization. If a model memorizes more, member losses move farther left relative to non-member losses, and membership inference gets easier. If regularization closes the train-test loss gap, the attack surface shrinks.

In [ ]:
lams_w = np.array([0.0, 0.001, 0.02, 0.2])
gaps_w, aucs_w = [], []
for lam_w in lams_w:
    th_w = fit_logreg_w(X_train_w, y_train_w, steps_w=2200, lr_w=0.25, lam_w=float(lam_w))
    tr_loss_w = log_loss_w(predict_proba_w(X_train_w, th_w), y_train_w)
    te_loss_w = log_loss_w(predict_proba_w(X_test_w, th_w), y_test_w)
    scores_w = np.r_[-tr_loss_w, -te_loss_w]
    labels_w = np.r_[np.ones_like(tr_loss_w), np.zeros_like(te_loss_w)]
    taus_w = np.linspace(scores_w.min(), scores_w.max(), 150)
    tpr_w, fpr_w = [], []
    for tau_w in taus_w:
        pred_w = (scores_w >= tau_w).astype(float)
        tpr_w.append(np.mean(pred_w[labels_w == 1] == 1))
        fpr_w.append(np.mean(pred_w[labels_w == 0] == 1))
    ord_w = np.argsort(fpr_w)
    gaps_w.append(float(te_loss_w.mean() - tr_loss_w.mean()))
    aucs_w.append(float(np.trapz(np.array(tpr_w)[ord_w], np.array(fpr_w)[ord_w])))

print("loss gaps:", np.round(gaps_w, 3))
print("attack AUCs:", np.round(aucs_w, 3))

assert max(aucs_w) > 0.53

▶ What you'll see: different regularization strengths produce different loss gaps and different attack AUCs.

In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.plot(lams_w, gaps_w, marker="o", color="crimson", label="test loss - train loss")
ax1.set_xscale("symlog", linthresh=0.001)
ax1.set_xlabel("regularization λ")
ax1.set_ylabel("loss gap", color="crimson")
ax2 = ax1.twinx()
ax2.plot(lams_w, aucs_w, marker="s", color="navy", label="attack AUC")
ax2.set_ylabel("attack AUC", color="navy")
plt.title("5: overfitting gap and attack strength")
plt.show()

▶ What you'll see: attack strength tracks the separation between member and non-member losses, though tiny samples make the curve imperfect.

*Why it's done this way:* Membership inference is not a separate magic failure; it is often the privacy face of generalization failure. Empirical risk minimization optimizes members directly, so the more the model's behavior differs between optimized points and fresh points, the more evidence an attacker gets. Regularization, early stopping, data augmentation, and differential privacy all matter because they reduce how uniquely a single row can shape the released model.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses
> small numbers, prints the intermediate values with inline `# ->` results, draws one picture,
> and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · True-label confidence becomes membership score

Membership attacks often start by turning model probabilities into per-example loss, then using
negative loss as a score where larger means more member-like.

In [ ]:
import numpy as np  # arrays and logarithms.
import matplotlib.pyplot as plt  # one picture per toy.

t1_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t1_prob = np.array([0.9, 0.7, 0.4, 0.2])  # predicted probability of class 1.

print("class-1 probabilities:", t1_prob.tolist())  # -> [0.9, 0.7, 0.4, 0.2]

t1_y = np.array([1, 1, 0, 0])  # true labels.

print("true labels:", t1_y.tolist())  # -> [1, 1, 0, 0]

t1_true_prob = t1_y * t1_prob + (1 - t1_y) * (1 - t1_prob)  # -> [0.9 0.7 0.6 0.8]

print("probability on true label:", np.round(t1_true_prob, 3).tolist())  # -> [0.9, 0.7, 0.6, 0.8]

t1_loss = -np.log(t1_true_prob)  # -> [0.10536052 0.35667494 0.51082562 0.22314355]

print("log losses:", np.round(t1_loss, 3).tolist())  # -> [0.105, 0.357, 0.511, 0.223]

t1_score = -t1_loss  # -> [-0.10536052 -0.35667494 -0.51082562 -0.22314355]

print("membership scores:", np.round(t1_score, 3).tolist())  # -> [-0.105, -0.357, -0.511, -0.223]

assert t1_score[0] > t1_score[2]

plt.figure(figsize=(4.6, 2.8))
plt.bar(range(len(t1_loss)), t1_loss, color="steelblue")
plt.xlabel("example")
plt.ylabel("loss")
plt.title("Toy 1 · lower loss means higher score")
plt.show()

▶ What you'll see: the most confident true-label example has the smallest loss and largest membership score.

### ✍️ Toy 2 · A loss-threshold attack has TPR, FPR, and advantage

Thresholding negative loss predicts member for scores above `tau`. Advantage is `TPR - FPR`, so it
rewards true detections and penalizes false alarms.

In [ ]:
import numpy as np  # arrays and rates.

t2_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t2_member_scores = np.array([-0.10, -0.20, -0.30, -0.40])  # scores from known members.

print("member scores:", t2_member_scores.tolist())  # -> [-0.1, -0.2, -0.3, -0.4]

t2_nonmember_scores = np.array([-0.25, -0.50, -0.70, -0.90])  # scores from known non-members.

print("non-member scores:", t2_nonmember_scores.tolist())  # -> [-0.25, -0.5, -0.7, -0.9]

t2_tau = -0.35  # attack threshold.

print("threshold tau:", round(t2_tau, 3))  # -> -0.35

t2_pred_member = t2_member_scores >= t2_tau  # -> [ True  True  True False]

print("member predictions:", t2_pred_member.astype(int).tolist())  # -> [1, 1, 1, 0]

t2_pred_nonmember = t2_nonmember_scores >= t2_tau  # -> [ True False False False]

print("non-member predictions:", t2_pred_nonmember.astype(int).tolist())  # -> [1, 0, 0, 0]

t2_tpr = float(t2_pred_member.mean())  # -> 0.75

print("TPR:", round(t2_tpr, 3))  # -> 0.75

t2_fpr = float(t2_pred_nonmember.mean())  # -> 0.25

print("FPR:", round(t2_fpr, 3))  # -> 0.25

t2_advantage = t2_tpr - t2_fpr  # -> 0.5

print("attack advantage:", round(t2_advantage, 3))  # -> 0.5

t2_accuracy = float((t2_pred_member.sum() + (~t2_pred_nonmember).sum()) / 8)  # -> 0.75

print("attack accuracy:", round(t2_accuracy, 3))  # -> 0.75

assert t2_advantage == 0.5

plt.figure(figsize=(4.6, 2.8))
plt.scatter(t2_member_scores, np.ones_like(t2_member_scores), label="members", color="seagreen")
plt.scatter(t2_nonmember_scores, np.zeros_like(t2_nonmember_scores), label="non-members", color="crimson")
plt.axvline(t2_tau, color="black", linestyle="--", label="tau")
plt.yticks([0, 1], ["non", "member"])
plt.xlabel("score = -loss")
plt.title("Toy 2 · threshold predicts membership")
plt.legend()
plt.show()

▶ What you'll see: three members and one non-member sit to the right of the threshold.

### ✍️ Toy 3 · Sweeping thresholds draws an ROC curve

ROC evaluates the same score rule at many thresholds. The area under that curve summarizes how
well scores rank members above non-members.

In [ ]:
import numpy as np  # arrays and trapezoids.

t3_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t3_scores = np.array([-0.10, -0.20, -0.30, -0.40, -0.25, -0.50, -0.70, -0.90])  # all attack scores.

print("all scores:", t3_scores.tolist())  # -> [-0.1, -0.2, -0.3, -0.4, -0.25, -0.5, -0.7, -0.9]

t3_labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])  # member labels.

print("member labels:", t3_labels.tolist())  # -> [1, 1, 1, 1, 0, 0, 0, 0]

t3_thresholds = np.array([-0.95, -0.65, -0.45, -0.35, -0.15])  # threshold sweep.

print("thresholds:", t3_thresholds.tolist())  # -> [-0.95, -0.65, -0.45, -0.35, -0.15]

t3_tpr = []  # true-positive rates.
t3_fpr = []  # false-positive rates.
for t3_tau in t3_thresholds:
    t3_pred = t3_scores >= t3_tau
    t3_tpr.append(float(t3_pred[t3_labels == 1].mean()))
    t3_fpr.append(float(t3_pred[t3_labels == 0].mean()))

print("TPR curve:", t3_tpr)  # -> [1.0, 1.0, 1.0, 0.75, 0.25]
print("FPR curve:", t3_fpr)  # -> [1.0, 0.5, 0.25, 0.25, 0.0]

t3_order = np.argsort(t3_fpr)  # order by false-positive rate.
t3_auc = float(np.trapz(np.array(t3_tpr)[t3_order], np.array(t3_fpr)[t3_order]))  # -> 0.875

print("ROC AUC:", round(t3_auc, 3))  # -> 0.875

assert t3_auc > 0.8

plt.figure(figsize=(4.2, 3.2))
plt.plot(np.array(t3_fpr)[t3_order], np.array(t3_tpr)[t3_order], marker="o", color="navy")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Toy 3 · threshold ROC")
plt.show()

▶ What you'll see: the ROC curve rises above the diagonal, giving AUC `0.875`.

### ✍️ Toy 4 · Shadow losses choose a transferable threshold

Shadow models provide labeled member and non-member scores for calibration. The chosen threshold is
then applied to the victim audit set.

In [ ]:
import numpy as np  # arrays and threshold search.

t4_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t4_shadow_in_loss = np.array([0.10, 0.20, 0.30, 0.35])  # shadow member losses.

print("shadow member losses:", t4_shadow_in_loss.tolist())  # -> [0.1, 0.2, 0.3, 0.35]

t4_shadow_out_loss = np.array([0.25, 0.50, 0.70, 0.90])  # shadow non-member losses.

print("shadow non-member losses:", t4_shadow_out_loss.tolist())  # -> [0.25, 0.5, 0.7, 0.9]

t4_shadow_scores = np.r_[-t4_shadow_in_loss, -t4_shadow_out_loss]  # negative losses.
t4_shadow_labels = np.r_[np.ones(4), np.zeros(4)]  # known shadow membership labels.
t4_thresholds = np.array([-0.8, -0.6, -0.4, -0.25, -0.15])  # candidate thresholds.

print("candidate thresholds:", t4_thresholds.tolist())  # -> [-0.8, -0.6, -0.4, -0.25, -0.15]

t4_advantages = []  # calibration advantages.
for t4_tau in t4_thresholds:
    t4_pred = t4_shadow_scores >= t4_tau
    t4_advantages.append(float(t4_pred[t4_shadow_labels == 1].mean() - t4_pred[t4_shadow_labels == 0].mean()))

print("shadow advantages:", t4_advantages)  # -> [0.25, 0.5, 0.75, 0.25, 0.25]

t4_tau = float(t4_thresholds[int(np.argmax(t4_advantages))])  # -> -0.4

print("shadow-chosen tau:", round(t4_tau, 3))  # -> -0.4

t4_victim_scores = np.array([-0.12, -0.22, -0.38, -0.42, -0.30, -0.55, -0.75, -0.95])  # victim audit scores.

print("victim scores:", t4_victim_scores.tolist())  # -> [-0.12, -0.22, -0.38, -0.42, -0.3, -0.55, -0.75, -0.95]

t4_victim_labels = np.r_[np.ones(4), np.zeros(4)]  # victim membership labels for audit scoring.
t4_victim_pred = t4_victim_scores >= t4_tau  # -> [ True  True  True False  True False False False]

print("victim predictions:", t4_victim_pred.astype(int).tolist())  # -> [1, 1, 1, 0, 1, 0, 0, 0]

t4_tpr = float(t4_victim_pred[t4_victim_labels == 1].mean())  # -> 0.75

print("victim TPR:", round(t4_tpr, 3))  # -> 0.75

t4_fpr = float(t4_victim_pred[t4_victim_labels == 0].mean())  # -> 0.25

print("victim FPR:", round(t4_fpr, 3))  # -> 0.25

assert t4_tpr - t4_fpr == 0.5

plt.figure(figsize=(4.6, 2.8))
plt.hist(-t4_shadow_in_loss, bins=4, alpha=0.65, label="shadow in", color="seagreen")
plt.hist(-t4_shadow_out_loss, bins=4, alpha=0.55, label="shadow out", color="crimson")
plt.axvline(t4_tau, color="black", linestyle="--", label="chosen tau")
plt.xlabel("score = -loss")
plt.title("Toy 4 · shadows calibrate tau")
plt.legend()
plt.show()

▶ What you'll see: the shadow threshold transfers to the victim and achieves `TPR - FPR = 0.5`.

### ✍️ Toy 5 · LiRA compares member and non-member likelihoods

A Gaussian likelihood-ratio score asks whether an observed score is more probable under the member
score distribution or the non-member distribution.

In [ ]:
import numpy as np  # arrays and Gaussian log densities.

t5_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t5_scores = np.array([-0.12, -0.22, -0.55, -0.75])  # audit scores.

print("audit scores:", t5_scores.tolist())  # -> [-0.12, -0.22, -0.55, -0.75]

t5_mu_in = -0.2  # member-score mean.

print("member mean:", round(t5_mu_in, 3))  # -> -0.2

t5_mu_out = -0.7  # non-member-score mean.

print("non-member mean:", round(t5_mu_out, 3))  # -> -0.7

t5_sd = 0.2  # shared standard deviation.

print("shared sd:", round(t5_sd, 3))  # -> 0.2

t5_log_in = -0.5 * np.log(2 * np.pi * t5_sd ** 2) - 0.5 * ((t5_scores - t5_mu_in) / t5_sd) ** 2  # member log densities.

print("member logpdf:", np.round(t5_log_in, 3).tolist())  # -> [0.61, 0.685, -0.841, -3.091]

t5_log_out = -0.5 * np.log(2 * np.pi * t5_sd ** 2) - 0.5 * ((t5_scores - t5_mu_out) / t5_sd) ** 2  # non-member log densities.

print("non-member logpdf:", np.round(t5_log_out, 3).tolist())  # -> [-3.515, -2.19, 0.409, 0.659]

t5_llr = t5_log_in - t5_log_out  # -> [ 4.125  2.875 -1.25  -3.75 ]

print("log likelihood ratios:", np.round(t5_llr, 3).tolist())  # -> [4.125, 2.875, -1.25, -3.75]

t5_member_like = t5_llr > 0  # -> [ True  True False False]

print("member-like flags:", t5_member_like.astype(int).tolist())  # -> [1, 1, 0, 0]

assert t5_member_like.tolist() == [True, True, False, False]

plt.figure(figsize=(4.6, 2.8))
plt.bar(range(len(t5_llr)), t5_llr, color=np.where(t5_member_like, "seagreen", "crimson"))
plt.axhline(0.0, color="black", linestyle="--")
plt.xlabel("example")
plt.ylabel("LLR")
plt.title("Toy 5 · positive LLR means member-like")
plt.show()

▶ What you'll see: the two high scores have positive likelihood ratios and are classified as member-like.

### ✍️ Toy 6 · Regularization shrinks the loss gap and attack AUC

When the train-test loss gap shrinks, membership scores become less separated. This toy treats
regularization strength as a privacy knob.

In [ ]:
import numpy as np  # arrays and differences.

t6_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t6_lambdas = np.array([0.0, 0.1, 1.0])  # regularization strengths.

print("regularization values:", t6_lambdas.tolist())  # -> [0.0, 0.1, 1.0]

t6_train_loss = np.array([0.20, 0.25, 0.35])  # member losses.

print("train losses:", t6_train_loss.tolist())  # -> [0.2, 0.25, 0.35]

t6_test_loss = np.array([0.50, 0.42, 0.39])  # non-member losses.

print("test losses:", t6_test_loss.tolist())  # -> [0.5, 0.42, 0.39]

t6_gap = t6_test_loss - t6_train_loss  # -> [0.3  0.17 0.04]

print("loss gaps:", np.round(t6_gap, 3).tolist())  # -> [0.3, 0.17, 0.04]

t6_attack_auc = np.array([0.75, 0.65, 0.55])  # attack strength by setting.

print("attack AUCs:", t6_attack_auc.tolist())  # -> [0.75, 0.65, 0.55]

t6_best_privacy = int(np.argmin(t6_gap))  # -> 2

print("smallest-gap setting index:", t6_best_privacy)  # -> 2

assert t6_attack_auc[t6_best_privacy] == t6_attack_auc.min()

plt.figure(figsize=(4.6, 2.8))
plt.plot(t6_lambdas, t6_gap, marker="o", label="loss gap")
plt.plot(t6_lambdas, t6_attack_auc, marker="s", label="attack AUC")
plt.xscale("symlog", linthresh=0.1)
plt.xlabel("regularization λ")
plt.ylabel("value")
plt.title("Toy 6 · smaller gap, weaker attack")
plt.legend()
plt.show()

▶ What you'll see: stronger regularization closes the loss gap and lowers the toy attack AUC.

### ✍️ Toy 7 · Class imbalance makes accuracy a bad privacy metric

If members are rare, a useless attack can get high accuracy by always saying non-member. TPR, FPR,
and advantage reveal that it learned nothing.

In [ ]:
import numpy as np  # arrays and rates.

t7_rng = np.random.default_rng(0)  # seeded generator for reproducible toys.
t7_member_labels = np.array([1, 1])  # two members.

print("member labels:", t7_member_labels.tolist())  # -> [1, 1]

t7_nonmember_labels = np.zeros(8, dtype=int)  # eight non-members.

print("non-member labels:", t7_nonmember_labels.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0]

t7_labels = np.r_[t7_member_labels, t7_nonmember_labels]  # combined labels.

print("all labels:", t7_labels.tolist())  # -> [1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

t7_attack_pred = np.zeros_like(t7_labels)  # always predict non-member.

print("attack predictions:", t7_attack_pred.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

t7_accuracy = float(np.mean(t7_attack_pred == t7_labels))  # -> 0.8

print("attack accuracy:", round(t7_accuracy, 3))  # -> 0.8

t7_tpr = float(np.mean(t7_attack_pred[t7_labels == 1] == 1))  # -> 0.0

print("TPR:", round(t7_tpr, 3))  # -> 0.0

t7_fpr = float(np.mean(t7_attack_pred[t7_labels == 0] == 1))  # -> 0.0

print("FPR:", round(t7_fpr, 3))  # -> 0.0

t7_advantage = t7_tpr - t7_fpr  # -> 0.0

print("attack advantage:", round(t7_advantage, 3))  # -> 0.0

assert t7_accuracy == 0.8 and t7_advantage == 0.0

plt.figure(figsize=(4.6, 2.8))
plt.bar(["accuracy", "TPR", "FPR", "advantage"], [t7_accuracy, t7_tpr, t7_fpr, t7_advantage], color=["gray", "seagreen", "crimson", "navy"])
plt.ylim(0, 1.05)
plt.ylabel("fraction")
plt.title("Toy 7 · high accuracy can be useless")
plt.show()

▶ What you'll see: the always-non-member attack scores `0.8` accuracy but has zero advantage.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, simulation, and small models built from scratch.
import matplotlib.pyplot as plt # load Matplotlib for loss histograms, ROC curves, and audit plots.
np.random.seed(0) # make the notebook's stochastic examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Turn model confidence into loss

**Goal.** Convert probabilities on true labels into per-example logistic losses, because membership attacks often start from unusually low training loss. We build it in 2 steps.

In [ ]:
p_b1 = np.array([0.95, 0.80, 0.55, 0.20]) # predicted probability assigned to label 1 for four examples.
y_b1 = np.array([1, 1, 0, 0]) # true labels for the same examples.

print("probabilities:", p_b1) # inspect the model scores before turning them into losses.
print("labels:", y_b1) # inspect the targets used to choose the correct probability.

In [ ]:
true_prob_b1 = np.where(y_b1 == 1, p_b1, 1 - p_b1) # select p for positive labels and 1-p for negative labels.
loss_b1 = -np.log(np.clip(true_prob_b1, 1e-12, 1.0)) # compute logistic loss on the true label.

print("true-label probabilities:", np.round(true_prob_b1, 3)) # inspect confidence on the correct class.
print("losses:", np.round(loss_b1, 3)) # inspect the attack score source.

assert round(float(loss_b1[0]), 3) == 0.051 # verify -log(0.95).
plt.figure(figsize=(4, 3)) # create a compact loss chart.
plt.bar(range(len(loss_b1)), loss_b1, color="teal") # show one loss per example.
plt.title("Basic 1: confidence becomes loss") # title the plot.
plt.xlabel("example") # label examples.
plt.ylabel("logistic loss") # label the loss scale.
plt.show() # display the chart.

▶ What you'll see: confident correct examples have tiny loss, while uncertain or wrong-looking examples have larger loss.

👀 Takeaway: a membership signal can come from the model being too confident on the true label.

### Basic 2 — Make a member/non-member score table

**Goal.** Build a labeled audit table with known membership labels, because attack evaluation needs both model scores and ground-truth membership. We build it in 2 steps.

In [ ]:
member_loss_b2 = np.array([0.05, 0.12, 0.18, 0.30, 0.42]) # toy losses on known training examples.
nonmember_loss_b2 = np.array([0.20, 0.35, 0.55, 0.80, 1.10]) # toy losses on held-out examples.
score_b2 = -np.r_[member_loss_b2, nonmember_loss_b2] # use negative loss so higher means more member-like.
m_b2 = np.r_[np.ones(len(member_loss_b2)), np.zeros(len(nonmember_loss_b2))] # 1=member, 0=non-member.

print("scores:", np.round(score_b2, 3)) # inspect the attack scores.
print("membership labels:", m_b2.astype(int)) # inspect the known labels.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact histogram.
plt.hist(-score_b2[m_b2 == 1], bins=5, alpha=0.65, label="members", color="seagreen") # show member losses.
plt.hist(-score_b2[m_b2 == 0], bins=5, alpha=0.55, label="non-members", color="crimson") # show non-member losses.
plt.title("Basic 2: labeled audit losses") # title the histogram.
plt.xlabel("loss") # label the loss axis.
plt.legend() # show group labels.
plt.show() # display the histogram.

▶ What you'll see: member losses are generally smaller, but the two groups overlap.

👀 Takeaway: membership inference is a binary classification problem over model-derived scores.

### Basic 3 — Apply one threshold rule

**Goal.** Implement `m_hat = 1[s >= τ]`, because the threshold attack is the simplest membership classifier. We build it in 2 steps.

In [ ]:
score_b3 = np.array([-0.05, -0.12, -0.18, -0.30, -0.42, -0.20, -0.35, -0.55, -0.80, -1.10]) # negative-loss scores.
m_b3 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # known membership labels for evaluation.
tau_b3 = -0.25 # classify examples with loss <= 0.25 as members.
pred_b3 = (score_b3 >= tau_b3).astype(int) # apply the membership rule.

print("predicted members:", pred_b3) # inspect the attack decisions.
print("true members:", m_b3) # inspect the ground truth.

In [ ]:
acc_b3 = np.mean(pred_b3 == m_b3) # compute raw attack accuracy.
tpr_b3 = np.mean(pred_b3[m_b3 == 1] == 1) # compute member detection rate.
fpr_b3 = np.mean(pred_b3[m_b3 == 0] == 1) # compute false alarms on non-members.

print("accuracy:", round(float(acc_b3), 3), "TPR:", round(float(tpr_b3), 3), "FPR:", round(float(fpr_b3), 3)) # inspect metrics.

assert round(float(acc_b3), 3) == 0.7 # verify this threshold's accuracy.
plt.figure(figsize=(4, 3)) # create a threshold plot.
plt.scatter(score_b3, m_b3, c=m_b3, cmap="coolwarm", s=70) # plot true membership by score.
plt.axvline(tau_b3, color="black", linestyle="--", label="τ") # show the threshold.
plt.title("Basic 3: threshold on score") # title the plot.
plt.xlabel("score = -loss") # label the score axis.
plt.ylabel("true membership") # label membership.
plt.legend() # show threshold label.
plt.show() # display the plot.

▶ What you'll see: examples to the right of the threshold are called members.

👀 Takeaway: the core attack is just a score, a threshold, and a binary decision.

### Basic 4 — Compute attack advantage

**Goal.** Measure `TPR - FPR`, because advantage reports how much better the attack is than false alarms. We build it in 2 steps.

In [ ]:
pred_b4 = np.array([1, 1, 1, 0, 0, 1, 0, 0, 0, 0]) # attack decisions from a toy threshold.
m_b4 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # true membership labels.
tp_b4 = np.sum((pred_b4 == 1) & (m_b4 == 1)) # count true positives.
fp_b4 = np.sum((pred_b4 == 1) & (m_b4 == 0)) # count false positives.

print("TP:", tp_b4, "FP:", fp_b4) # inspect numerator counts.

In [ ]:
tpr_b4 = tp_b4 / np.sum(m_b4 == 1) # member recall.
fpr_b4 = fp_b4 / np.sum(m_b4 == 0) # non-member false alarm rate.
adv_b4 = tpr_b4 - fpr_b4 # membership advantage.

print("TPR:", round(float(tpr_b4), 3), "FPR:", round(float(fpr_b4), 3), "advantage:", round(float(adv_b4), 3)) # inspect the privacy metric.

assert round(float(adv_b4), 3) == 0.4 # verify 0.6 - 0.2.
plt.figure(figsize=(4, 3)) # create a metric comparison chart.
plt.bar(["TPR", "FPR", "adv"], [tpr_b4, fpr_b4, adv_b4], color=["seagreen", "crimson", "navy"]) # show the pieces.
plt.title("Basic 4: attack advantage") # title the chart.
plt.ylim(0, 1) # keep the scale interpretable.
plt.show() # display the bars.

▶ What you'll see: advantage subtracts false alarms from member hits.

👀 Takeaway: privacy audits prefer advantage over accuracy when membership labels may be imbalanced.

### Basic 5 — Sweep thresholds and draw an ROC curve

**Goal.** Evaluate every threshold, because one cutoff hides the full privacy tradeoff between true positives and false positives. We build it in 3 steps.

In [ ]:
score_b5 = np.array([-0.05, -0.12, -0.18, -0.30, -0.42, -0.20, -0.35, -0.55, -0.80, -1.10]) # negative-loss scores.
m_b5 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # ground-truth membership labels.
taus_b5 = np.linspace(score_b5.min(), score_b5.max(), 50) # candidate thresholds.

print("threshold count:", len(taus_b5)) # inspect the sweep size.

In [ ]:
tpr_b5, fpr_b5 = [], [] # prepare ROC arrays.
for tau_b5 in taus_b5: # evaluate one threshold at a time.
    pred_b5 = (score_b5 >= tau_b5).astype(int) # call high-score examples members.
    tpr_b5.append(np.mean(pred_b5[m_b5 == 1] == 1)) # member hit rate.
    fpr_b5.append(np.mean(pred_b5[m_b5 == 0] == 1)) # non-member false alarm rate.
order_b5 = np.argsort(fpr_b5) # sort points from low to high FPR for AUC.
auc_b5 = float(np.trapz(np.array(tpr_b5)[order_b5], np.array(fpr_b5)[order_b5])) # approximate ROC area.

print("AUC:", round(auc_b5, 3)) # inspect threshold-free attack strength.

assert auc_b5 > 0.7 # verify clear leakage in the toy scores.

In [ ]:
plt.figure(figsize=(4, 3)) # create an ROC figure.
plt.plot(np.array(fpr_b5)[order_b5], np.array(tpr_b5)[order_b5], marker="o", color="purple") # draw ROC curve.
plt.plot([0, 1], [0, 1], "--", color="gray") # random-guess baseline.
plt.title("Basic 5: ROC sweep") # title the plot.
plt.xlabel("false positive rate") # label x-axis.
plt.ylabel("true positive rate") # label y-axis.
plt.show() # display the ROC curve.

▶ What you'll see: the curve rises above the diagonal when scores separate members from non-members.

👀 Takeaway: ROC summarizes all possible threshold attacks on the same score.

### Basic 6 — See why overfitting creates separation

**Goal.** Compare train and test losses directly, because the train-test gap is the attacker's raw material. We build it in 2 steps.

In [ ]:
train_loss_b6 = np.array([0.04, 0.08, 0.12, 0.20, 0.35, 0.50]) # losses on optimized training examples.
test_loss_b6 = np.array([0.15, 0.28, 0.40, 0.62, 0.90, 1.20]) # losses on fresh examples.
gap_b6 = float(test_loss_b6.mean() - train_loss_b6.mean()) # compute the generalization gap in loss.

print("train mean:", round(float(train_loss_b6.mean()), 3), "test mean:", round(float(test_loss_b6.mean()), 3)) # inspect means.
print("loss gap:", round(gap_b6, 3)) # inspect separation.

In [ ]:
plt.figure(figsize=(4, 3)) # create side-by-side bars.
plt.boxplot([train_loss_b6, test_loss_b6], labels=["train", "test"]) # summarize distributions.
plt.title("Basic 6: train-test loss gap") # title the plot.
plt.ylabel("loss") # label the loss scale.
plt.show() # display boxplots.
assert gap_b6 > 0.3 # verify a noticeable gap.

▶ What you'll see: training losses sit lower than test losses, giving the attacker a statistical cue.

👀 Takeaway: membership inference often measures overfitting from the privacy side.

### Basic 7 — Calibrate a threshold on shadow data

**Goal.** Choose `τ` from separate shadow losses, because the attack threshold should not be tuned on victim membership labels. We build it in 2 steps.

In [ ]:
shadow_in_b7 = np.array([0.05, 0.10, 0.18, 0.22, 0.31, 0.40]) # known member losses from shadow models.
shadow_out_b7 = np.array([0.16, 0.25, 0.38, 0.55, 0.72, 0.95]) # known non-member losses from shadows.
shadow_score_b7 = -np.r_[shadow_in_b7, shadow_out_b7] # convert to high-is-member scores.
shadow_m_b7 = np.r_[np.ones(len(shadow_in_b7)), np.zeros(len(shadow_out_b7))] # labels for calibration only.

print("shadow score range:", round(float(shadow_score_b7.min()), 3), "to", round(float(shadow_score_b7.max()), 3)) # inspect range.

In [ ]:
taus_b7 = np.linspace(shadow_score_b7.min(), shadow_score_b7.max(), 60) # candidate thresholds.
advs_b7 = [] # store shadow advantages.
for tau_b7 in taus_b7: # sweep thresholds.
    pred_b7 = (shadow_score_b7 >= tau_b7).astype(int) # classify with current threshold.
    advs_b7.append(np.mean(pred_b7[shadow_m_b7 == 1] == 1) - np.mean(pred_b7[shadow_m_b7 == 0] == 1)) # compute advantage.
best_tau_b7 = float(taus_b7[int(np.argmax(advs_b7))]) # select threshold by shadow advantage.

print("shadow-selected tau:", round(best_tau_b7, 3)) # inspect chosen threshold.

plt.figure(figsize=(4, 3)) # create calibration curve.
plt.plot(taus_b7, advs_b7, color="navy") # show advantage by threshold.
plt.axvline(best_tau_b7, color="red", linestyle="--") # mark selected threshold.
plt.title("Basic 7: shadow threshold calibration") # title the plot.
plt.xlabel("τ") # label threshold axis.
plt.ylabel("shadow advantage") # label metric.
plt.show() # display curve.

▶ What you'll see: one shadow threshold maximizes member/non-member separation on calibration data.

👀 Takeaway: shadow data lets the attacker tune the decision rule without peeking at victim labels.

### Basic 8 — Compute one likelihood ratio

**Goal.** Compare how likely one score is under member and non-member distributions, because LiRA is a likelihood-ratio attack. We build it in 2 steps.

In [ ]:
score_b8 = -0.12 # one candidate example's score, where larger means more member-like.
mu_in_b8, sd_in_b8 = -0.18, 0.10 # simple Gaussian model for member scores.
mu_out_b8, sd_out_b8 = -0.45, 0.20 # simple Gaussian model for non-member scores.

print("score:", score_b8) # inspect the candidate score.

In [ ]:
logpdf_in_b8 = -0.5 * np.log(2 * np.pi * sd_in_b8 ** 2) - 0.5 * ((score_b8 - mu_in_b8) / sd_in_b8) ** 2 # member log density.
logpdf_out_b8 = -0.5 * np.log(2 * np.pi * sd_out_b8 ** 2) - 0.5 * ((score_b8 - mu_out_b8) / sd_out_b8) ** 2 # non-member log density.
llr_b8 = logpdf_in_b8 - logpdf_out_b8 # log likelihood ratio.

print("log p(score|member):", round(float(logpdf_in_b8), 3)) # inspect member evidence.
print("log p(score|non-member):", round(float(logpdf_out_b8), 3)) # inspect non-member evidence.
print("LLR:", round(float(llr_b8), 3)) # inspect final evidence score.

assert llr_b8 > 0 # verify this score is more member-like.

In [ ]:
like_in_b8 = np.exp(logpdf_in_b8) # convert member log evidence to likelihood.
like_out_b8 = np.exp(logpdf_out_b8) # convert non-member log evidence to likelihood.
ratio_b8 = like_in_b8 / like_out_b8 # compute the likelihood ratio.
plt.figure(figsize=(4, 3)) # create likelihood comparison chart.
plt.bar(["member", "non-member"], [like_in_b8, like_out_b8], color=["teal", "gray"]) # compare densities at the score.
plt.text(0.5, max(like_in_b8, like_out_b8) * 0.9, f"ratio = {ratio_b8:.2f}×", ha="center") # annotate evidence ratio.
plt.title("Basic 8: likelihood at one score") # title the plot.
plt.ylabel("density at score") # label likelihood scale.
plt.show() # display chart.

▶ What you'll see: the score is more likely under the member Gaussian, so the LLR is positive.

👀 Takeaway: LiRA replaces a raw score with evidence: member likelihood divided by non-member likelihood.

### Basic 9 — Measure leakage under class imbalance

**Goal.** Compare accuracy and advantage when there are many non-members, because raw accuracy can hide a weak or trivial attack. We build it in 2 steps.

In [ ]:
m_b9 = np.r_[np.ones(10), np.zeros(90)] # imbalanced audit set with many more non-members.
pred_all_out_b9 = np.zeros_like(m_b9) # trivial attack that calls everyone non-member.
acc_all_out_b9 = np.mean(pred_all_out_b9 == m_b9) # raw accuracy of the trivial rule.
tpr_all_out_b9 = np.mean(pred_all_out_b9[m_b9 == 1] == 1) # member detection rate.
fpr_all_out_b9 = np.mean(pred_all_out_b9[m_b9 == 0] == 1) # false alarm rate.

print("trivial accuracy:", round(float(acc_all_out_b9), 3)) # inspect misleading accuracy.
print("trivial advantage:", round(float(tpr_all_out_b9 - fpr_all_out_b9), 3)) # inspect real attack value.

In [ ]:
pred_some_b9 = m_b9.copy() # create a stronger toy attack for comparison.
pred_some_b9[20:40] = 1 # add false positives among non-members.
acc_some_b9 = np.mean(pred_some_b9 == m_b9) # compute raw accuracy.
adv_some_b9 = np.mean(pred_some_b9[m_b9 == 1] == 1) - np.mean(pred_some_b9[m_b9 == 0] == 1) # compute advantage.

print("stronger accuracy:", round(float(acc_some_b9), 3), "stronger advantage:", round(float(adv_some_b9), 3)) # compare metrics.

plt.figure(figsize=(4, 3)) # create a metric chart.
plt.bar(["trivial acc", "trivial adv", "attack acc", "attack adv"], [acc_all_out_b9, 0, acc_some_b9, adv_some_b9], color=["gray", "gray", "teal", "teal"]) # show mismatch.
plt.xticks(rotation=20) # rotate labels for readability.
plt.title("Basic 9: imbalance changes metric meaning") # title the plot.
plt.show() # display chart.

▶ What you'll see: the trivial all-non-member rule has high accuracy but zero advantage.

👀 Takeaway: privacy leakage is about distinguishing members, not exploiting label imbalance.

### Basic 10 — Inspect a loss cutoff on real toy data

**Goal.** Generate a tiny train/test split and plot a cutoff, because visual inspection catches whether an attack is driven by real separation or noise. We build it in 3 steps.

In [ ]:
rng_b10 = np.random.default_rng(10) # local randomness for reproducibility.
member_loss_b10 = rng_b10.gamma(shape=1.5, scale=0.18, size=40) # synthetic low member losses.
nonmember_loss_b10 = rng_b10.gamma(shape=1.8, scale=0.28, size=120) # synthetic broader non-member losses.
cut_b10 = 0.28 # call examples with loss at or below this cutoff members.

print("member mean loss:", round(float(member_loss_b10.mean()), 3)) # inspect member losses.
print("non-member mean loss:", round(float(nonmember_loss_b10.mean()), 3)) # inspect non-member losses.

In [ ]:
pred_member_b10 = member_loss_b10 <= cut_b10 # member decisions for true members.
pred_nonmember_b10 = nonmember_loss_b10 <= cut_b10 # member decisions for true non-members.
tpr_b10 = float(pred_member_b10.mean()) # true positive rate.
fpr_b10 = float(pred_nonmember_b10.mean()) # false positive rate.

print("TPR:", round(tpr_b10, 3), "FPR:", round(fpr_b10, 3), "advantage:", round(tpr_b10 - fpr_b10, 3)) # inspect cutoff behavior.

assert tpr_b10 > fpr_b10 # verify the cutoff contains a membership signal.

In [ ]:
plt.figure(figsize=(5, 3)) # create loss histogram.
plt.hist(member_loss_b10, bins=18, alpha=0.65, label="members", color="seagreen") # plot member losses.
plt.hist(nonmember_loss_b10, bins=18, alpha=0.5, label="non-members", color="crimson") # plot non-member losses.
plt.axvline(cut_b10, color="black", linestyle="--", label="loss cutoff") # show threshold.
plt.title("Basic 10: inspect the cutoff") # title plot.
plt.xlabel("loss") # label x-axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: the cutoff slices through overlapping histograms; good audits report both hits and false alarms.

👀 Takeaway: always inspect member and non-member score distributions before trusting one privacy number.

## 🟡 Easy

### Easy 1 — Train a victim logistic model from scratch

**Goal.** Fit a NumPy-only classifier and measure the train-test loss gap, because membership inference needs a victim model with observable scores. We build it in 4 steps.

In [ ]:
rng_e1 = np.random.default_rng(21) # local generator for reproducible data.
y_train_e1 = rng_e1.integers(0, 2, size=70).astype(float) # training labels.
y_test_e1 = rng_e1.integers(0, 2, size=300).astype(float) # held-out labels.
X_train_e1 = np.where(y_train_e1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e1.normal(size=(70, 2)) # train features.
X_test_e1 = np.where(y_test_e1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e1.normal(size=(300, 2)) # test features.

print("train/test:", X_train_e1.shape, X_test_e1.shape) # inspect data sizes.

In [ ]:
X1_train_e1 = np.c_[np.ones(len(X_train_e1)), X_train_e1] # add intercept to training features.
theta_e1 = np.zeros(3) # initialize logistic regression weights.
for step_e1 in range(2200): # run gradient descent.
    p_e1 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_e1, -30, 30))) # current train probabilities.
    grad_e1 = X1_train_e1.T @ (p_e1 - y_train_e1) / len(y_train_e1) + 0.001 * np.r_[0, theta_e1[1:]] # regularized gradient.
    theta_e1 -= 0.25 * grad_e1 # update weights.

print("theta:", np.round(theta_e1, 3)) # inspect learned model.

In [ ]:
p_train_e1 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_e1, -30, 30))) # train probabilities.
p_test_e1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_test_e1)), X_test_e1] @ theta_e1, -30, 30))) # test probabilities.
loss_train_e1 = -(y_train_e1 * np.log(np.clip(p_train_e1, 1e-12, 1)) + (1 - y_train_e1) * np.log(np.clip(1 - p_train_e1, 1e-12, 1))) # train losses.
loss_test_e1 = -(y_test_e1 * np.log(np.clip(p_test_e1, 1e-12, 1)) + (1 - y_test_e1) * np.log(np.clip(1 - p_test_e1, 1e-12, 1))) # test losses.

print("mean train loss:", round(float(loss_train_e1.mean()), 3), "mean test loss:", round(float(loss_test_e1.mean()), 3)) # inspect gap.

assert loss_train_e1.mean() < loss_test_e1.mean() # verify member losses are lower in this toy run.

In [ ]:
plt.figure(figsize=(5, 3)) # create histogram figure.
plt.hist(loss_train_e1, bins=18, alpha=0.65, label="train/member", color="seagreen") # plot member losses.
plt.hist(loss_test_e1, bins=18, alpha=0.55, label="test/non-member", color="crimson") # plot non-member losses.
plt.title("Easy 1: victim loss distributions") # title plot.
plt.xlabel("logistic loss") # label axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: a train/test loss gap even though both sets come from the same population.

👀 Takeaway: the victim's per-example loss is enough to start a membership audit.

### Easy 2 — Build a loss-threshold attack on the victim

**Goal.** Select the best threshold on labeled audit data and report advantage, because this is the canonical baseline attack. We build it in 3 steps.

In [ ]:
score_e2 = -np.r_[loss_train_e1, loss_test_e1] # high score means low loss and more member-like.
m_e2 = np.r_[np.ones(len(loss_train_e1)), np.zeros(len(loss_test_e1))] # known labels for this audit experiment.
taus_e2 = np.linspace(score_e2.min(), score_e2.max(), 200) # sweep candidate thresholds.

print("audit examples:", len(score_e2)) # inspect total audit size.

In [ ]:
best_tau_e2, best_adv_e2, best_acc_e2 = None, -1, None # initialize best-threshold tracking.
for tau_e2 in taus_e2: # evaluate candidate thresholds.
    pred_e2 = (score_e2 >= tau_e2).astype(float) # threshold attack decision.
    tpr_e2 = np.mean(pred_e2[m_e2 == 1] == 1) # member hit rate.
    fpr_e2 = np.mean(pred_e2[m_e2 == 0] == 1) # false alarm rate.
    adv_e2 = tpr_e2 - fpr_e2 # attack advantage.
    if adv_e2 > best_adv_e2: # keep the best advantage threshold.
        best_tau_e2, best_adv_e2, best_acc_e2 = tau_e2, adv_e2, np.mean(pred_e2 == m_e2) # store results.

print("best tau:", round(float(best_tau_e2), 3), "accuracy:", round(float(best_acc_e2), 3), "advantage:", round(float(best_adv_e2), 3)) # inspect attack.

assert best_adv_e2 > 0.05 # verify nontrivial leakage.

In [ ]:
plt.figure(figsize=(5, 3)) # create threshold plot.
plt.hist(score_e2[m_e2 == 1], bins=18, alpha=0.65, label="members", color="seagreen") # plot member scores.
plt.hist(score_e2[m_e2 == 0], bins=18, alpha=0.55, label="non-members", color="crimson") # plot non-member scores.
plt.axvline(best_tau_e2, color="black", linestyle="--", label="best τ") # mark selected threshold.
plt.title("Easy 2: loss-threshold attack") # title plot.
plt.xlabel("score = -loss") # label axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: the threshold is placed where high member density starts to separate from non-member density.

👀 Takeaway: the baseline attack is simple, auditable, and often surprisingly strong.

### Easy 3 — Calibrate with shadow models and transfer to the victim

**Goal.** Train shadow classifiers to choose a threshold, because real attackers should not tune on victim membership labels. We build it in 4 steps.

In [ ]:
def one_shadow_e3(seed_e3): # train one shadow model and return member/non-member losses.
    rng_e3 = np.random.default_rng(seed_e3) # local generator.
    y_in_e3 = rng_e3.integers(0, 2, size=70).astype(float) # shadow train labels.
    y_out_e3 = rng_e3.integers(0, 2, size=180).astype(float) # shadow holdout labels.
    X_in_e3 = np.where(y_in_e3[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e3.normal(size=(70, 2)) # shadow train features.
    X_out_e3 = np.where(y_out_e3[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e3.normal(size=(180, 2)) # shadow holdout features.
    X1_e3 = np.c_[np.ones(len(X_in_e3)), X_in_e3] # add intercept.
    th_e3 = np.zeros(3) # initialize weights.
    for _ in range(1200): # train logistic model.
        p_e3 = 1 / (1 + np.exp(-np.clip(X1_e3 @ th_e3, -30, 30))) # probabilities.
        th_e3 -= 0.25 * (X1_e3.T @ (p_e3 - y_in_e3) / len(y_in_e3) + 0.001 * np.r_[0, th_e3[1:]]) # gradient step.
    pin_e3 = 1 / (1 + np.exp(-np.clip(X1_e3 @ th_e3, -30, 30))) # member probabilities.
    pout_e3 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_out_e3)), X_out_e3] @ th_e3, -30, 30))) # non-member probabilities.
    lin_e3 = -(y_in_e3 * np.log(np.clip(pin_e3, 1e-12, 1)) + (1-y_in_e3) * np.log(np.clip(1-pin_e3, 1e-12, 1))) # member losses.
    lout_e3 = -(y_out_e3 * np.log(np.clip(pout_e3, 1e-12, 1)) + (1-y_out_e3) * np.log(np.clip(1-pout_e3, 1e-12, 1))) # non-member losses.
    return lin_e3, lout_e3

print("shadow helper ready") # confirm function definition.

In [ ]:
shadow_in_e3, shadow_out_e3 = [], [] # collect losses across shadows.
for seed_e3 in range(31, 37): # train several independent shadow models.
    si_e3, so_e3 = one_shadow_e3(seed_e3) # get losses from one shadow.
    shadow_in_e3.append(si_e3); shadow_out_e3.append(so_e3) # store losses.
shadow_in_e3 = np.concatenate(shadow_in_e3); shadow_out_e3 = np.concatenate(shadow_out_e3) # flatten arrays.

print("shadow losses:", shadow_in_e3.shape[0], shadow_out_e3.shape[0]) # inspect calibration size.

In [ ]:
shadow_score_e3 = -np.r_[shadow_in_e3, shadow_out_e3] # high score means likely member.
shadow_m_e3 = np.r_[np.ones(len(shadow_in_e3)), np.zeros(len(shadow_out_e3))] # known shadow labels.
taus_e3 = np.linspace(shadow_score_e3.min(), shadow_score_e3.max(), 180) # threshold candidates.
advs_e3 = [] # store shadow advantages.
for tau_e3 in taus_e3: # sweep thresholds.
    pred_e3 = (shadow_score_e3 >= tau_e3).astype(float) # classify shadow examples.
    advs_e3.append(np.mean(pred_e3[shadow_m_e3 == 1] == 1) - np.mean(pred_e3[shadow_m_e3 == 0] == 1)) # advantage.
tau_shadow_e3 = float(taus_e3[int(np.argmax(advs_e3))]) # choose best shadow threshold.

print("shadow tau:", round(tau_shadow_e3, 3)) # inspect transferred threshold.

In [ ]:
victim_pred_e3 = (score_e2 >= tau_shadow_e3).astype(float) # apply shadow threshold to victim audit scores.
tpr_e3 = np.mean(victim_pred_e3[m_e2 == 1] == 1) # victim member hit rate.
fpr_e3 = np.mean(victim_pred_e3[m_e2 == 0] == 1) # victim false alarm rate.

print("victim TPR:", round(float(tpr_e3), 3), "FPR:", round(float(fpr_e3), 3), "advantage:", round(float(tpr_e3 - fpr_e3), 3)) # inspect transfer result.

assert tpr_e3 - fpr_e3 > 0.0 # verify positive transfer in this toy run.
plt.figure(figsize=(5, 3)) # create calibration histogram.
plt.hist(shadow_in_e3, bins=18, alpha=0.65, label="shadow members", color="seagreen") # shadow member losses.
plt.hist(shadow_out_e3, bins=18, alpha=0.55, label="shadow non-members", color="crimson") # shadow non-member losses.
plt.axvline(-tau_shadow_e3, color="black", linestyle="--", label="loss cutoff") # cutoff in loss units.
plt.title("Easy 3: shadow-model calibration") # title plot.
plt.xlabel("loss") # label axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: shadow losses choose a cutoff that still gives positive advantage on the victim.

👀 Takeaway: shadow models make membership inference a transferable calibration problem.

### Easy 4 — Implement a Gaussian LiRA score

**Goal.** Fit member and non-member score distributions from shadow models and compute a likelihood-ratio score on victim examples. We build it in 3 steps.

In [ ]:
mu_in_e4 = float((-shadow_in_e3).mean()) # mean member score from shadows.
raw_sd_in_e4 = float((-shadow_in_e3).std()) # raw standard deviation of member scores.
mu_out_e4 = float((-shadow_out_e3).mean()) # mean non-member score from shadows.
raw_sd_out_e4 = float((-shadow_out_e3).std()) # raw standard deviation of non-member scores.
sd_pool_e4 = float(np.sqrt((raw_sd_in_e4 ** 2 + raw_sd_out_e4 ** 2) / 2)) # pooled scale for a small Gaussian demo.
sd_in_e4, sd_out_e4 = sd_pool_e4, sd_pool_e4 # use equal variance so the LLR mainly reflects mean separation.

print("in μ/sd:", round(mu_in_e4, 3), round(sd_in_e4, 3), "out μ/sd:", round(mu_out_e4, 3), round(sd_out_e4, 3)) # inspect fitted distributions.

assert mu_in_e4 > mu_out_e4 # verify member scores are higher on average.

In [ ]:
def logpdf_e4(x_e4, mu_e4, sd_e4): # normal log density helper.
    sd_e4 = max(float(sd_e4), 1e-6) # avoid divide by zero.
    return -0.5 * np.log(2 * np.pi * sd_e4 ** 2) - 0.5 * ((x_e4 - mu_e4) / sd_e4) ** 2 # return log N(x; mu, sd).
llr_e4 = logpdf_e4(score_e2, mu_in_e4, sd_in_e4) - logpdf_e4(score_e2, mu_out_e4, sd_out_e4) # LiRA-style evidence.

print("mean member LLR:", round(float(llr_e4[m_e2 == 1].mean()), 3), "mean non-member LLR:", round(float(llr_e4[m_e2 == 0].mean()), 3)) # inspect separation.

assert llr_e4[m_e2 == 1].mean() > llr_e4[m_e2 == 0].mean() # verify member-like evidence.

In [ ]:
taus_e4 = np.linspace(llr_e4.min(), llr_e4.max(), 200) # threshold candidate LLRs.
tpr_e4, fpr_e4 = [], [] # ROC arrays.
for tau_e4 in taus_e4: # sweep LLR thresholds.
    pred_e4 = (llr_e4 >= tau_e4).astype(float) # high LLR means member.
    tpr_e4.append(np.mean(pred_e4[m_e2 == 1] == 1)) # hit rate.
    fpr_e4.append(np.mean(pred_e4[m_e2 == 0] == 1)) # false alarm rate.
order_e4 = np.argsort(fpr_e4) # sort by FPR.
auc_e4 = float(np.trapz(np.array(tpr_e4)[order_e4], np.array(fpr_e4)[order_e4])) # compute AUC.

print("LiRA-style AUC:", round(auc_e4, 3)) # inspect attack strength.

assert auc_e4 > 0.52 # verify nontrivial likelihood-ratio attack.
plt.figure(figsize=(4, 3)) # create ROC plot.
plt.plot(np.array(fpr_e4)[order_e4], np.array(tpr_e4)[order_e4], color="purple") # draw LiRA ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Easy 4: LiRA-style ROC") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.show() # display ROC.

▶ What you'll see: a likelihood-ratio attack curve based on shadow-estimated score distributions.

👀 Takeaway: LiRA asks which distribution makes the observed score more plausible.

### Easy 5 — Compare regularization as a privacy control

**Goal.** Sweep logistic-regression regularization and compare loss gaps with attack AUC, because reducing overfitting often reduces membership leakage. We build it in 3 steps.

In [ ]:
lams_e5 = np.array([0.0, 0.001, 0.02, 0.2]) # regularization strengths to compare.
gaps_e5, aucs_e5 = [], [] # store privacy diagnostics.

print("lambda grid:", lams_e5) # inspect sweep values.

In [ ]:
for lam_e5 in lams_e5: # train one victim per lambda.
    theta_loop_e5 = np.zeros(3) # reset weights.
    for step_e5 in range(1800): # train logistic regression.
        p_loop_e5 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_loop_e5, -30, 30))) # train probabilities.
        grad_loop_e5 = X1_train_e1.T @ (p_loop_e5 - y_train_e1) / len(y_train_e1) + lam_e5 * np.r_[0, theta_loop_e5[1:]] # gradient.
        theta_loop_e5 -= 0.25 * grad_loop_e5 # update.
    ptr_e5 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_loop_e5, -30, 30))) # train probabilities.
    pte_e5 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_test_e1)), X_test_e1] @ theta_loop_e5, -30, 30))) # test probabilities.
    ltr_e5 = -(y_train_e1 * np.log(np.clip(ptr_e5, 1e-12, 1)) + (1-y_train_e1) * np.log(np.clip(1-ptr_e5, 1e-12, 1))) # train losses.
    lte_e5 = -(y_test_e1 * np.log(np.clip(pte_e5, 1e-12, 1)) + (1-y_test_e1) * np.log(np.clip(1-pte_e5, 1e-12, 1))) # test losses.
    scores_loop_e5 = -np.r_[ltr_e5, lte_e5] # attack scores.
    labels_loop_e5 = np.r_[np.ones(len(ltr_e5)), np.zeros(len(lte_e5))] # membership labels.
    taus_loop_e5 = np.linspace(scores_loop_e5.min(), scores_loop_e5.max(), 120) # thresholds.
    tprs_loop_e5, fprs_loop_e5 = [], [] # ROC arrays.
    for tau_loop_e5 in taus_loop_e5: # evaluate thresholds.
        pred_loop_e5 = (scores_loop_e5 >= tau_loop_e5).astype(float) # decisions.
        tprs_loop_e5.append(np.mean(pred_loop_e5[labels_loop_e5 == 1] == 1)) # TPR.
        fprs_loop_e5.append(np.mean(pred_loop_e5[labels_loop_e5 == 0] == 1)) # FPR.
    order_loop_e5 = np.argsort(fprs_loop_e5) # sort ROC points.
    gaps_e5.append(float(lte_e5.mean() - ltr_e5.mean())) # store loss gap.
    aucs_e5.append(float(np.trapz(np.array(tprs_loop_e5)[order_loop_e5], np.array(fprs_loop_e5)[order_loop_e5]))) # store AUC.

print("gaps:", np.round(gaps_e5, 3)) # inspect gaps.
print("AUCs:", np.round(aucs_e5, 3)) # inspect attacks.

assert max(aucs_e5) > 0.53 # verify at least one nontrivial attack.

In [ ]:
plt.figure(figsize=(5, 3)) # create sweep plot.
plt.plot(lams_e5, gaps_e5, marker="o", label="loss gap", color="crimson") # plot generalization gap.
plt.plot(lams_e5, aucs_e5, marker="s", label="attack AUC", color="navy") # plot attack strength.
plt.xscale("symlog", linthresh=0.001) # show zero and small lambdas cleanly.
plt.title("Easy 5: regularization and leakage") # title plot.
plt.xlabel("λ") # label x-axis.
plt.legend() # show line labels.
plt.show() # display plot.

▶ What you'll see: the privacy signal changes when the model's regularization changes.

👀 Takeaway: privacy audits belong in model selection, not only after deployment.

## 🔴 Advanced

### Advanced 1 — Run a full shadow-model attack pipeline

**Goal.** Package the shadow-model workflow end to end, because a privacy red-team needs repeatable calibration and victim evaluation. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(101) # reproducible victim data.
y_vtrain_a1 = rng_a1.integers(0, 2, size=90).astype(float) # victim train labels.
y_vtest_a1 = rng_a1.integers(0, 2, size=350).astype(float) # victim non-member labels.
X_vtrain_a1 = np.where(y_vtrain_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 1.0 * rng_a1.normal(size=(90, 2)) # victim training features.
X_vtest_a1 = np.where(y_vtest_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 1.0 * rng_a1.normal(size=(350, 2)) # victim test features.

print("victim data:", X_vtrain_a1.shape, X_vtest_a1.shape) # inspect sizes.

In [ ]:
X1_vtrain_a1 = np.c_[np.ones(len(X_vtrain_a1)), X_vtrain_a1] # add intercept.
th_v_a1 = np.zeros(3) # initialize victim weights.
for step_a1 in range(2200): # train victim model.
    p_a1 = 1 / (1 + np.exp(-np.clip(X1_vtrain_a1 @ th_v_a1, -30, 30))) # probabilities.
    th_v_a1 -= 0.22 * (X1_vtrain_a1.T @ (p_a1 - y_vtrain_a1) / len(y_vtrain_a1) + 0.001 * np.r_[0, th_v_a1[1:]]) # update.
pin_v_a1 = 1 / (1 + np.exp(-np.clip(X1_vtrain_a1 @ th_v_a1, -30, 30))) # victim member probabilities.
pout_v_a1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_vtest_a1)), X_vtest_a1] @ th_v_a1, -30, 30))) # victim non-member probabilities.
loss_in_v_a1 = -(y_vtrain_a1*np.log(np.clip(pin_v_a1,1e-12,1)) + (1-y_vtrain_a1)*np.log(np.clip(1-pin_v_a1,1e-12,1))) # victim member losses.
loss_out_v_a1 = -(y_vtest_a1*np.log(np.clip(pout_v_a1,1e-12,1)) + (1-y_vtest_a1)*np.log(np.clip(1-pout_v_a1,1e-12,1))) # victim non-member losses.

print("victim loss gap:", round(float(loss_out_v_a1.mean() - loss_in_v_a1.mean()), 3)) # inspect overfitting signal.

In [ ]:
shadow_in_a1, shadow_out_a1 = [], [] # shadow calibration storage.
for seed_a1 in range(120, 130): # run several shadows.
    rng_s_a1 = np.random.default_rng(seed_a1) # local generator.
    ys_in_a1 = rng_s_a1.integers(0, 2, size=90).astype(float) # shadow train labels.
    ys_out_a1 = rng_s_a1.integers(0, 2, size=220).astype(float) # shadow holdout labels.
    Xs_in_a1 = np.where(ys_in_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + rng_s_a1.normal(size=(90, 2)) # shadow train features.
    Xs_out_a1 = np.where(ys_out_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + rng_s_a1.normal(size=(220, 2)) # shadow holdout features.
    X1s_a1 = np.c_[np.ones(len(Xs_in_a1)), Xs_in_a1] # add intercept.
    th_s_a1 = np.zeros(3) # initialize.
    for _ in range(1200): # train shadow.
        ps_a1 = 1 / (1 + np.exp(-np.clip(X1s_a1 @ th_s_a1, -30, 30))) # probabilities.
        th_s_a1 -= 0.22 * (X1s_a1.T @ (ps_a1 - ys_in_a1) / len(ys_in_a1) + 0.001 * np.r_[0, th_s_a1[1:]]) # update.
    ps_in_a1 = 1 / (1 + np.exp(-np.clip(X1s_a1 @ th_s_a1, -30, 30))) # in probabilities.
    ps_out_a1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(Xs_out_a1)), Xs_out_a1] @ th_s_a1, -30, 30))) # out probabilities.
    shadow_in_a1.append(-(ys_in_a1*np.log(np.clip(ps_in_a1,1e-12,1)) + (1-ys_in_a1)*np.log(np.clip(1-ps_in_a1,1e-12,1)))) # append in losses.
    shadow_out_a1.append(-(ys_out_a1*np.log(np.clip(ps_out_a1,1e-12,1)) + (1-ys_out_a1)*np.log(np.clip(1-ps_out_a1,1e-12,1)))) # append out losses.
shadow_in_a1 = np.concatenate(shadow_in_a1); shadow_out_a1 = np.concatenate(shadow_out_a1) # flatten losses.

print("shadow calibration size:", len(shadow_in_a1), len(shadow_out_a1)) # inspect sizes.

In [ ]:
score_shadow_a1 = -np.r_[shadow_in_a1, shadow_out_a1] # shadow scores.
label_shadow_a1 = np.r_[np.ones(len(shadow_in_a1)), np.zeros(len(shadow_out_a1))] # shadow membership labels.
taus_a1 = np.linspace(score_shadow_a1.min(), score_shadow_a1.max(), 180) # threshold candidates.
advs_a1 = [np.mean((score_shadow_a1[label_shadow_a1==1] >= t_a1)) - np.mean((score_shadow_a1[label_shadow_a1==0] >= t_a1)) for t_a1 in taus_a1] # shadow advantages.
tau_a1 = float(taus_a1[int(np.argmax(advs_a1))]) # selected shadow threshold.
score_victim_a1 = -np.r_[loss_in_v_a1, loss_out_v_a1] # victim scores.
label_victim_a1 = np.r_[np.ones(len(loss_in_v_a1)), np.zeros(len(loss_out_v_a1))] # victim labels for evaluation.
pred_victim_a1 = (score_victim_a1 >= tau_a1).astype(float) # transfer attack.
adv_victim_a1 = np.mean(pred_victim_a1[label_victim_a1==1] == 1) - np.mean(pred_victim_a1[label_victim_a1==0] == 1) # victim advantage.

print("shadow tau:", round(tau_a1, 3), "victim advantage:", round(float(adv_victim_a1), 3)) # inspect final audit.

assert adv_victim_a1 > 0.0 # verify positive leakage.
plt.figure(figsize=(5, 3)) # plot victim losses.
plt.hist(loss_in_v_a1, bins=22, alpha=0.65, label="victim members", color="seagreen") # member histogram.
plt.hist(loss_out_v_a1, bins=22, alpha=0.55, label="victim non-members", color="crimson") # non-member histogram.
plt.axvline(-tau_a1, color="black", linestyle="--", label="shadow cutoff") # threshold in loss units.
plt.title("Advanced 1: transferred shadow attack") # title plot.
plt.xlabel("loss") # label axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: a threshold learned from shadow models applied unchanged to the victim model's loss histogram.

👀 Takeaway: an operational membership audit separates calibration data from victim evaluation.

### Advanced 2 — Compare global-threshold and LiRA-style attacks

**Goal.** Run two attacks on the same victim scores, because likelihood ratios can use distribution shape beyond a single global cutoff. We build it in 3 steps.

In [ ]:
mu_in_a2 = float((-shadow_in_a1).mean()) # member score mean.
raw_sd_in_a2 = float((-shadow_in_a1).std()) # raw member score std.
mu_out_a2 = float((-shadow_out_a1).mean()) # non-member score mean.
raw_sd_out_a2 = float((-shadow_out_a1).std()) # raw non-member score std.
sd_pool_a2 = float(np.sqrt((raw_sd_in_a2 ** 2 + raw_sd_out_a2 ** 2) / 2)) # pooled Gaussian scale.
sd_in_a2, sd_out_a2 = sd_pool_a2, sd_pool_a2 # equal variance makes the likelihood ratio a calibrated separation score.

print("Gaussian score models:", np.round([mu_in_a2, sd_in_a2, mu_out_a2, sd_out_a2], 3)) # inspect fitted distributions.

In [ ]:
def logpdf_a2(x_a2, mu_a2, sd_a2): # normal log density.
    sd_a2 = max(float(sd_a2), 1e-6) # avoid zero scale.
    return -0.5*np.log(2*np.pi*sd_a2**2) - 0.5*((x_a2-mu_a2)/sd_a2)**2 # log density.
llr_a2 = logpdf_a2(score_victim_a1, mu_in_a2, sd_in_a2) - logpdf_a2(score_victim_a1, mu_out_a2, sd_out_a2) # likelihood-ratio score.

print("mean LLR members/non-members:", round(float(llr_a2[label_victim_a1==1].mean()), 3), round(float(llr_a2[label_victim_a1==0].mean()), 3)) # inspect separation.

In [ ]:
def auc_from_scores_a2(scores_a2, labels_a2): # simple ROC AUC helper.
    taus_local_a2 = np.linspace(scores_a2.min(), scores_a2.max(), 220) # thresholds.
    tpr_local_a2, fpr_local_a2 = [], [] # curves.
    for tau_local_a2 in taus_local_a2: # sweep thresholds.
        pred_local_a2 = (scores_a2 >= tau_local_a2).astype(float) # predictions.
        tpr_local_a2.append(np.mean(pred_local_a2[labels_a2==1] == 1)) # TPR.
        fpr_local_a2.append(np.mean(pred_local_a2[labels_a2==0] == 1)) # FPR.
    order_local_a2 = np.argsort(fpr_local_a2) # sort points.
    return float(np.trapz(np.array(tpr_local_a2)[order_local_a2], np.array(fpr_local_a2)[order_local_a2])), np.array(fpr_local_a2)[order_local_a2], np.array(tpr_local_a2)[order_local_a2] # return AUC and ROC.
auc_loss_a2, fpr_loss_a2, tpr_loss_a2 = auc_from_scores_a2(score_victim_a1, label_victim_a1) # global score attack.
auc_lira_a2, fpr_lira_a2, tpr_lira_a2 = auc_from_scores_a2(llr_a2, label_victim_a1) # LiRA-style attack.

print("loss AUC:", round(auc_loss_a2, 3), "LiRA AUC:", round(auc_lira_a2, 3)) # compare attacks.

assert auc_lira_a2 > 0.52 and auc_loss_a2 > 0.52 # verify both attacks are meaningful.
plt.figure(figsize=(4.8, 3.5)) # create comparison ROC.
plt.plot(fpr_loss_a2, tpr_loss_a2, label=f"loss AUC={auc_loss_a2:.3f}", color="navy") # loss ROC.
plt.plot(fpr_lira_a2, tpr_lira_a2, label=f"LiRA AUC={auc_lira_a2:.3f}", color="purple") # LLR ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Advanced 2: attack ROC comparison") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: two ROC curves from the same victim model; either can be stronger depending on calibration quality.

👀 Takeaway: LiRA reframes membership inference as evidence under two score distributions.

### Advanced 3 — Audit per-class leakage

**Goal.** Break attack strength down by label, because one average privacy score can hide that a rare or harder class leaks more. We build it in 3 steps.

In [ ]:
labels_for_scores_a3 = np.r_[y_vtrain_a1, y_vtest_a1] # true class labels aligned to victim membership scores.
classes_a3 = np.array([0.0, 1.0]) # binary classes in the toy problem.

print("class counts:", [int(np.sum(labels_for_scores_a3 == c_a3)) for c_a3 in classes_a3]) # inspect class balance.

In [ ]:
class_auc_a3 = [] # store AUC per class.
for c_a3 in classes_a3: # evaluate each class separately.
    mask_c_a3 = labels_for_scores_a3 == c_a3 # select examples with this true label.
    auc_c_a3, _, _ = auc_from_scores_a2(score_victim_a1[mask_c_a3], label_victim_a1[mask_c_a3]) # compute score attack AUC within class.
    class_auc_a3.append(auc_c_a3) # save class AUC.

print("per-class AUC:", np.round(class_auc_a3, 3)) # inspect subgroup privacy leakage.

assert max(class_auc_a3) > 0.55 # verify at least one class leaks clearly in this toy run.

In [ ]:
plt.figure(figsize=(4, 3)) # create per-class leakage chart.
plt.bar(["class 0", "class 1"], class_auc_a3, color=["darkorange", "teal"]) # plot class AUCs.
plt.axhline(0.5, color="gray", linestyle="--", label="random") # mark no-leakage baseline.
plt.title("Advanced 3: per-class leakage") # title plot.
plt.ylabel("membership AUC") # label axis.
plt.legend() # show baseline label.
plt.show() # display bars.

▶ What you'll see: class-specific AUCs may differ even when the overall AUC looks acceptable.

👀 Takeaway: privacy audits should slice by meaningful groups, not only report one aggregate score.

### Advanced 4 — Use confidence margin instead of loss

**Goal.** Attack with a different score, the confidence margin, because deployed APIs may expose probabilities but not labels or losses. We build it in 3 steps.

In [ ]:
prob_true_in_a4 = np.where(y_vtrain_a1 == 1, pin_v_a1, 1 - pin_v_a1) # victim confidence on true label for members.
prob_true_out_a4 = np.where(y_vtest_a1 == 1, pout_v_a1, 1 - pout_v_a1) # victim confidence on true label for non-members.
margin_score_a4 = np.r_[prob_true_in_a4 - (1 - prob_true_in_a4), prob_true_out_a4 - (1 - prob_true_out_a4)] # true-class margin as attack score.

print("mean member margin:", round(float(margin_score_a4[label_victim_a1==1].mean()), 3)) # inspect member confidence.
print("mean non-member margin:", round(float(margin_score_a4[label_victim_a1==0].mean()), 3)) # inspect non-member confidence.

In [ ]:
auc_margin_a4, fpr_margin_a4, tpr_margin_a4 = auc_from_scores_a2(margin_score_a4, label_victim_a1) # ROC from margin score.
auc_loss_a4, fpr_loss_a4, tpr_loss_a4 = auc_from_scores_a2(score_victim_a1, label_victim_a1) # ROC from loss score.

print("margin AUC:", round(auc_margin_a4, 3), "loss AUC:", round(auc_loss_a4, 3)) # compare score choices.

assert auc_margin_a4 > 0.50 # verify confidence margins leak in this toy setting.

In [ ]:
plt.figure(figsize=(4.8, 3.5)) # create score comparison ROC.
plt.plot(fpr_margin_a4, tpr_margin_a4, label=f"margin AUC={auc_margin_a4:.3f}", color="teal") # margin ROC.
plt.plot(fpr_loss_a4, tpr_loss_a4, label=f"loss AUC={auc_loss_a4:.3f}", color="navy") # loss ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Advanced 4: score choice matters") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: confidence margin and loss contain related but not identical membership signals.

👀 Takeaway: audit the scores an API actually releases, not only the score convenient for researchers.

### Advanced 5 — Simulate a release gate with attack advantage

**Goal.** Turn the audit into a deployment decision, because privacy testing should create an actionable release gate. We build it in 3 steps.

In [ ]:
models_a5 = ["small gap", "medium gap", "large gap"] # toy candidate models.
member_means_a5 = np.array([0.35, 0.25, 0.12]) # smaller member losses for increasingly overfit models.
nonmember_mean_a5 = 0.42 # common non-member average loss.
rng_a5 = np.random.default_rng(505) # reproducible simulation.

print("candidate models:", models_a5) # inspect candidates.

In [ ]:
advantages_a5 = [] # store attack advantage per model.
for mean_a5 in member_means_a5: # simulate one model's losses.
    mem_loss_a5 = rng_a5.gamma(shape=2.0, scale=mean_a5 / 2.0, size=120) # member losses.
    non_loss_a5 = rng_a5.gamma(shape=2.0, scale=nonmember_mean_a5 / 2.0, size=600) # non-member losses.
    scores_a5 = -np.r_[mem_loss_a5, non_loss_a5] # attack scores.
    labels_a5 = np.r_[np.ones(len(mem_loss_a5)), np.zeros(len(non_loss_a5))] # membership labels.
    taus_a5 = np.linspace(scores_a5.min(), scores_a5.max(), 120) # thresholds.
    adv_grid_a5 = [] # advantage over thresholds.
    for tau_a5 in taus_a5: # sweep threshold.
        pred_a5 = (scores_a5 >= tau_a5).astype(float) # classify members.
        adv_grid_a5.append(np.mean(pred_a5[labels_a5==1] == 1) - np.mean(pred_a5[labels_a5==0] == 1)) # advantage.
    advantages_a5.append(float(max(adv_grid_a5))) # store worst/best attacker advantage.

print("advantages:", np.round(advantages_a5, 3)) # inspect privacy risk.

assert advantages_a5[2] > advantages_a5[0] # verify larger gap produces stronger attack.

In [ ]:
gate_a5 = 0.25 # deployment policy: maximum allowed attack advantage.
passes_a5 = np.array(advantages_a5) <= gate_a5 # release decisions.

print("passes gate:", dict(zip(models_a5, passes_a5.tolist()))) # inspect decisions.

plt.figure(figsize=(5, 3)) # create gate plot.
plt.bar(models_a5, advantages_a5, color=np.where(passes_a5, "seagreen", "crimson")) # green pass, red fail.
plt.axhline(gate_a5, color="black", linestyle="--", label="release gate") # show policy threshold.
plt.title("Advanced 5: privacy release gate") # title plot.
plt.ylabel("attack advantage") # label axis.
plt.legend() # show gate label.
plt.show() # display plot.

▶ What you'll see: the most overfit candidate fails the privacy gate even if it might look attractive on training metrics.

👀 Takeaway: membership inference turns privacy risk into a number that can block or approve model release.